In [2]:
## ============================================
# CARGA BCRA_INFO A TABLAS EXISTENTES (FORZANDO TSV)
# - Periodo sin "d" (ej: 202505) -> carpeta 202505d
# - Lee todos los archivos como TSV (tab + comillas)
# - Asigna nombres de columnas EXACTAMENTE como en la tabla (si coincide el número)
# - Si la primera fila es encabezado, la descarta y reintenta
# - Busca el archivo más grande si no hay "COMPLETO"
# ============================================

# !pip install -q pandas sqlalchemy pymysql chardet

from pathlib import Path
from datetime import datetime
import pandas as pd
import chardet, re
from sqlalchemy import create_engine, text

# ---------- CONFIG ----------
BASE_DIR   = Path(r"C:\Users\Usuario\Downloads")

from getpass import getpass

MYSQL_USER = "root"
MYSQL_PASS = getpass("🔒 Ingresá la contraseña de MySQL: ")
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
DB_NAME    = "bcra_info"

# Carpeta → archivo “especial” si no hay "completo"
SPECIAL_FILE_BY_FOLDER = {
    "bal_hist": "balhist.txt",
    "cuentas" : "cuentas.txt",
}

# Carpetas a omitir si hiciera falta (puedes dejar vacío)
SKIP_SECTIONS = set()
# --------------------------------------------

# ===== INPUT PERÍODO (sin 'd') =====
PERIODO = input("Ingresá el período a cargar (ej: 202505): ").strip()
PERIODO_DIR = PERIODO + "d"
PATH_TEC_CONT = BASE_DIR / PERIODO_DIR / "Entfin" / "Tec_Cont"
if not PATH_TEC_CONT.exists():
    raise FileNotFoundError(f"No se encontró la carpeta: {PATH_TEC_CONT}")

fecha_carga = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"✅ Carpeta detectada: {PATH_TEC_CONT}")
print(f"👉 Periodo a grabar en base: {PERIODO}\n")

# ===== CONEXIÓN A MYSQL =====
engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASS}@{MYSQL_HOST}:{MYSQL_PORT}/{DB_NAME}?charset=utf8mb4"
)

# ===== HELPERS =====
def detect_encoding(path: Path) -> str:
    try:
        raw = path.read_bytes()[:8192]
        return chardet.detect(raw).get("encoding") or "latin1"
    except Exception:
        return "latin1"

def get_table_columns(table_name: str):
    with engine.begin() as conn:
        rows = conn.execute(
            text("""
                SELECT COLUMN_NAME
                FROM information_schema.COLUMNS
                WHERE TABLE_SCHEMA = :s AND TABLE_NAME = :t
                ORDER BY ORDINAL_POSITION
            """),
            {"s": DB_NAME, "t": table_name}
        ).fetchall()
    return [r[0] for r in rows]  # orden real en MySQL

def read_tsv_force(path: Path) -> pd.DataFrame:
    enc = detect_encoding(path)
    # Forzamos TSV con comillas; todo como texto
    df = pd.read_csv(
        path,
        sep="\t",
        quotechar='"',
        header=None,   # tratamos todo como “sin encabezado” y resolvemos luego
        dtype=str,
        engine="python",
        encoding=enc,
        on_bad_lines="skip"
    )
    # limpieza
    for c in df.columns:
        df[c] = df[c].astype(str).str.strip()
    return df

def coerce_columns_to_table(df: pd.DataFrame, table: str) -> pd.DataFrame:
    """
    Intenta poner a df los nombres EXACTOS de la tabla (sin Periodo/Fecha_Carga).
    - Si la cantidad de columnas coincide => renombra y retorna.
    - Si la primera fila parece encabezado y al quitarla coincide => quita fila 0 y renombra.
    """
    cols_db = get_table_columns(table)
    data_cols_db = [c for c in cols_db if c.lower() not in ("periodo", "fecha_carga")]
    n_db = len(data_cols_db)
    n_df = df.shape[1]

    # Caso directo
    if n_df == n_db:
        df.columns = data_cols_db
        return df

    # Caso con encabezado en la primera fila
    if n_df == n_db:
        return df  # ya cubierto

    # Si df tiene mismas columnas que la tabla pero con una FILA de encabezado:
    # chequear si la fila 0 contiene letras y coincide en forma general
    first_row = df.iloc[0].astype(str).str.lower().tolist()
    looks_header = sum(1 for x in first_row if re.search(r"[a-z_]", x)) >= max(1, int(0.5 * n_df))
    if looks_header:
        df2 = df.iloc[1:].reset_index(drop=True)
        if df2.shape[1] == n_db:
            df2.columns = data_cols_db
            return df2

    # Si hay una columna de más que es un “separador” vacío al final, intentar drop:
    if n_df == n_db + 1 and (df.iloc[:, -1] == "").all():
        df2 = df.iloc[:, :-1]
        df2.columns = data_cols_db
        return df2

    # No se pudo casar por cantidad -> dejamos genéricas y que falle arriba si no matchea
    df.columns = [f"col_{i+1}" for i in range(n_df)]
    return df

def pick_data_file(section: Path) -> Path:
    """
    1) 'completo' en el nombre (cualquier extensión) -> usar
    2) archivo “especial” definido por carpeta -> usar
    3) archivo MÁS GRANDE de la carpeta, ignorando nombres tipo 'formato', 'dicc', 'layout'
       y aceptando archivos SIN extensión.
    """
    # 1) 'completo'
    for f in section.iterdir():
        if f.is_file() and "completo" in f.name.lower():
            return f

    # 2) especial por carpeta
    sec = section.name.lower()
    if sec in SPECIAL_FILE_BY_FOLDER:
        cand = section / SPECIAL_FILE_BY_FOLDER[sec]
        if cand.exists() and cand.is_file():
            return cand

    # 3) mayor archivo válido (no formato/dicc/layout)
    candidates = []
    for f in section.iterdir():
        if not f.is_file():
            continue
        n = f.name.lower()
        if any(x in n for x in ("formato", "dicc", "layout")):
            continue
        if f.stat().st_size > 0:
            candidates.append((f.stat().st_size, f))
    if candidates:
        candidates.sort(reverse=True)
        return candidates[0][1]
    return None

# ===== PROCESO =====
ok, fail, omit = 0, 0, 0

for section in sorted([p for p in PATH_TEC_CONT.iterdir() if p.is_dir()]):
    table = section.name.lower()

    print(f"\n➡️ Entrando a carpeta: {section.name}")
    
    if table in SKIP_SECTIONS:
        print(f"[SKIP  ] {table:20s} (omitida por configuración)")
        omit += 1
        continue

    data_file = pick_data_file(section)
    if not data_file:
        print(f"[OMITIDA] {table:20s} (no encontré archivo de datos)")
        omit += 1
        continue

    try:
       # 1) Omitir si no existe la tabla
        cols_db = get_table_columns(table)

        if len(cols_db) == 0:
            print(f"[OMITIDA] {table:20s} (tabla no existe en la base)")
            omit += 1
            continue
        
        # 2) Leer TSV forzado
        df = read_tsv_force(data_file)

                     
        # 3) Asignar nombres EXACTOS de la tabla (si las cuentas dan)
        df = coerce_columns_to_table(df, table)

        # 4) Si no hay columnas de la tabla, esto dejaría 'col_1'...'col_n';
        #    chequeamos para evitar subir basura:
        cols_db = [c for c in cols_db if c.lower() not in ("periodo","fecha_carga")]
        if set(df.columns) != set(cols_db):
            # si difiere por orden pero misma cantidad, intentamos reordenar:
            if len(df.columns) == len(cols_db):
                df.columns = cols_db  # forzamos nombres por posición
            else:
                raise ValueError(f"El archivo '{data_file.name}' no matchea cantidad de columnas de {table}: "
                                 f"{df.shape[1]} vs {len(cols_db)}")

        if df.empty:
            print(f"[VACÍA ] {table:20s} (0 filas en archivo)")
            continue

        # 5) Agregar control
        df["Periodo"] = PERIODO
        df["Fecha_Carga"] = fecha_carga

        # 6) Insertar
        df.to_sql(table, engine, if_exists="append", index=False, chunksize=5000, method="multi")
        print(f"[OK    ] {table:20s} -> {len(df):,} filas insertadas  |  archivo: {data_file.name}")
        ok += 1

    except Exception as e:
        print(f"[ERROR ] {table:20s} -> {e}")
        fail += 1

print(f"\n✅ Terminado. Secciones OK: {ok} | Errores: {fail} | Omitidas: {omit}")

Ingresá el período a cargar (ej: 202505):  202512


✅ Carpeta detectada: C:\Users\Usuario\Downloads\202512d\Entfin\Tec_Cont
👉 Periodo a grabar en base: 202512


➡️ Entrando a carpeta: bal_hist
[OMITIDA] bal_hist             (tabla no existe en la base)

➡️ Entrando a carpeta: baldet
[OK    ] baldet               -> 27,204 filas insertadas  |  archivo: COMPLETO.TXT

➡️ Entrando a carpeta: balres
[OK    ] balres               -> 5,073 filas insertadas  |  archivo: COMPLETO.TXT

➡️ Entrando a carpeta: cta_impu
[OK    ] cta_impu             -> 27,203 filas insertadas  |  archivo: completo.txt

➡️ Entrando a carpeta: cuentas
[OK    ] cuentas              -> 5,518 filas insertadas  |  archivo: cuentas.txt

➡️ Entrando a carpeta: distribgeo
[OMITIDA] distribgeo           (tabla no existe en la base)

➡️ Entrando a carpeta: entidad
[OK    ] entidad              -> 85 filas insertadas  |  archivo: COMPLETO.TXT

➡️ Entrando a carpeta: entint
[OK    ] entint               -> 275 filas insertadas  |  archivo: completo.txt

➡️ Entrando a carpeta: es

KeyError: "['dolar'] not in index"